## Results Summary\n\n

In [ ]:
# ==================== 2. CONFUSION MATRICES ====================
if BENCHMARK_RESULTS_PATH.exists():
    print("\n

In [ ]:
# ==================== 5. LIVE INFERENCE DEMO ====================
demo_audio = "sample_visec.wav"

if not Path(demo_audio).exists():
    print(f"Demo file {demo_audio} not found. Please provide an audio file.")
else:
    print(f"Running Inference on {demo_audio}\n")
    import IPython.display as ipd
    from IPython.display import display
    display(ipd.Audio(demo_audio))

    # ECAPA Inference
    from ECAPA.predict_emotion import EmotionClassifier
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_labels = len(emotion_labels)
    
    ecapa_model = EmotionClassifier(num_labels).to(device)
    checkpoint_path = Path("ECAPA/emotion_model/best_ecapa_model.pth")
    
    if checkpoint_path.exists():
        ecapa_model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
        ecapa_model.eval()
        
        audio, sr = librosa.load(demo_audio, sr=16000)
        from transformers import AutoFeatureExtractor
        processor = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
        inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
        features = inputs.input_values.to(device)
        
        with torch.no_grad():
            outputs = ecapa_model(features)
            probs = torch.softmax(outputs, dim=1)
            pred_idx = torch.argmax(probs, dim=1).item()
            conf = probs[0, pred_idx].item()
            
        print(f"\n